# 04. 대화형 Agent 완성

---
## 0. 설치 & 경로

In [ ]:
!pip install -U duckduckgo-search

In [ ]:
!pip install openai python-dotenv langchain langchain-openai langchain-core langchain-community langchain-text-splitters langchain-classic faiss-cpu pypdf pytz yfinance requests ddgs tabulate rank_bm25

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

RAG_SAMPLES = os.path.join(os.getcwd(), 'samples', 'pdf_samples')
print('RAG_SAMPLES:', RAG_SAMPLES)
for p in sorted(Path(RAG_SAMPLES).glob('*.pdf')):
    print(' -', p.name)

---
## Step 1. RAG VectorStore 

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import os

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# 학칙.pdf
loader = PyPDFLoader(os.path.join(RAG_SAMPLES, '학칙.pdf'))
all_splits = text_splitter.split_documents(loader.load())

# news.pdf
loader_news = PyPDFLoader(os.path.join(RAG_SAMPLES, '보도자료.pdf'))
news_splits = text_splitter.split_documents(loader_news.load())
all_splits.extend(news_splits)

embedding = OpenAIEmbeddings(
    model='text-embedding-3-large', api_key=OPENAI_API_KEY
)

persist_directory = 'chroma_store'	

# 저장된 크로마 DB가 없다면 새로 만들기
if not os.path.exists(persist_directory):
    print("Creating new Chroma store")
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory
    )

else:
    print("Loading existing Chroma store")
    vectorstore = Chroma(		
        persist_directory=persist_directory, 
        embedding_function=embedding
    )
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

In [ ]:
retriever # 생성한 VectorStore 기반으로 검색을 수행하는 Retriever 객체

---
## Step 1.1 검색 품질 점검

`석사학위과정 수업연한`처럼 **별표(부록)에 같은 단어가 반복**되는 PDF에서는, 기본 유사도 검색이 엉뚱한 청크 3개만 가져올 수 있습니다.  
아래에서 검색 결과와 **실제 정답 조항**을 비교해 봅니다.

In [ ]:
demo_query = '석사학위과정 수업연한'

retriever_baseline = vectorstore.as_retriever(search_kwargs={'k': 3})
baseline_hits = retriever_baseline.invoke(demo_query)

print(f'질문: {demo_query}\n')
for i, doc in enumerate(baseline_hits, 1):
    has_answer = '수업연한' in doc.page_content and '2년(4학기)' in doc.page_content
    print(f'[{i}] 수업연한 정답 포함: {has_answer}')
    print(doc.page_content[:350].replace('\n', ' '))
    print('-' * 60)

In [ ]:
# 학칙.pdf 본문에 실제로 있는 조항 (검색이 찾아야 할 정답)
answer_chunks = [
    s for s in all_splits
    if '수업연한' in s.page_content and '2년(4학기)' in s.page_content
]

print(f'정답 청크 수: {len(answer_chunks)}')
print('=' * 60)
print(answer_chunks[0].page_content[:700])

---
## Step 1.2 BM25 키워드 검색 + 하이브리드

**BM25**는 단어(키워드) 빈도 기반 검색입니다. `수업연한`처럼 **문서에 정확히 등장하는 용어**를 찾을 때 벡터 검색보다 유리한 경우가 많습니다.

| 방식 | 장점 | 약점 |
|------|------|------|
| 벡터 검색 | 의미가 비슷한 문장도 찾음 | 별표처럼 같은 단어가 반복되면 엉뚱한 청크가 올라옴 |
| BM25 | 키워드가 들어 있는 청크를 직접 찾음 | 한글 PDF는 띄어쓰기가 적어 **토크나이저 설정**이 필요 |

실습에서는 **BM25 + 벡터**를 `EnsembleRetriever`로 합쳐 하이브리드 검색을 씁니다.

In [ ]:
import re
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.tools import create_retriever_tool


def ko_preprocess(text):
    """한글 학칙 PDF용 — 띄어쓰기 대신 한글 어절 단위로 토큰 분리"""
    return re.findall(r'[가-힣]+|\d+', text)


def show_hits(label, hits, preview=3):
    print(f'\n=== {label} ===')
    for i, doc in enumerate(hits[:preview], 1):
        has_answer = '수업연한' in doc.page_content and '2년(4학기)' in doc.page_content
        print(f'[{i}] 수업연한 정답 포함: {has_answer}')
        print(doc.page_content[:280].replace('\n', ' '))
        print('-' * 50)

In [ ]:
# BM25 키워드 검색
bm25_retriever = BM25Retriever.from_documents(all_splits, preprocess_func=ko_preprocess)
bm25_retriever.k = 3
bm25_hits = bm25_retriever.invoke(demo_query)
show_hits('BM25 키워드 검색 (k=3)', bm25_hits)

In [ ]:
# 하이브리드: BM25 + 벡터
vector_retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)
hybrid_hits = hybrid_retriever.invoke(demo_query)
show_hits('하이브리드 BM25 + 벡터 (k=3)', hybrid_hits)

found = any(
    '수업연한' in d.page_content and '2년(4학기)' in d.page_content
    for d in hybrid_hits
)
print('\n수업연한 정답 청크 포함 여부:', found)

---
## Step 2. Retriever → Tool (5.3 마지막 섹션)

`langchain_core.tools.create_retriever_tool` — 5.3과 동일합니다.

> **Step 1.2**에서 하이브리드 retriever로 `search_pdf_tool`을 이미 만들었다면, 이 셀은 생략해도 됩니다.

In [ ]:
from langchain_core.tools import create_retriever_tool

search_pdf_tool = create_retriever_tool(
    retriever,
    name='search_pdf_documents',
    description='pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.',
)

---
## Step 3. 유틸 Tool (5.2 복습)

시각 · 주가 — `@tool` 로 등록합니다.

In [ ]:
from datetime import datetime
import pytz
import yfinance as yf
from langchain_core.tools import tool

@tool
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각. timezone 예: Asia/Seoul, location 예: 서울"""
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime('%Y-%m-%d %H:%M:%S')
    return f'{timezone} ({location}) 현재시각 {now}'

@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """주식 가격 조회. ticker 예: TSLA, period 예: 1mo"""
    history = yf.Ticker(ticker).history(period=period)
    return history.tail(3).to_string() if not history.empty else '데이터 없음'


In [ ]:
print(get_current_time.invoke({'timezone': 'Asia/Seoul', 'location': '서울'}))

---
## Step 4. DuckDuckGo 웹 검색

최신 뉴스·시사는 PDF RAG가 아니라 **웹 검색** Tool을 씁니다.  
Agent가 질문마다 `search_pdf_documents` vs `duckduckgo_results_json` 중 선택합니다.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchResults

web_search = DuckDuckGoSearchResults(num_results=3)
print(web_search.name, '-', web_search.description[:80], '...')

In [ ]:
web_search.invoke("26년 월드컵 한국 vs 멕시코 점수는?")

---
## Step 5. AgentExecutor — Tool 통합

5.2 수동 tool 루프 대신 **AgentExecutor**가 tool_calls 루프를 자동 처리합니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

agent_tools = [get_current_time, get_yf_stock_history, search_pdf_tool, web_search]
print([t.name for t in agent_tools])

prompt = ChatPromptTemplate.from_messages([
    ('system', '''너는 사용자를 돕는 AI입니다. 질문에 맞는 도구를 골라 사용하세요.

- samples/rag_samples PDF (학칙, news 등) → search_pdf_documents
- 최신 뉴스·시사·웹 정보 → duckduckgo_results_json
- 주가 → get_yf_stock_history
- 현재 시각 → get_current_time

도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

In [ ]:
# PDF → search_pdf_documents
r = executor.invoke({'input': '석사학위과정 수업연한은?', 'chat_history': []})
print('RAG:', r['output'])

In [ ]:
search_pdf_tool = create_retriever_tool(
    hybrid_retriever,
    name='search_pdf_documents',
    description='pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.',
)
# PDF → search_pdf_documents

agent_tools = [get_current_time, get_yf_stock_history, search_pdf_tool, web_search]
agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

r = executor.invoke({'input': '석사학위과정 수업연한은?', 'chat_history': []})
print('RAG:', r['output'])

In [ ]:
# 시각 → get_current_time
r = executor.invoke({'input': '부산 지금 몇시야?', 'chat_history': []})
print('시각:', r['output'])

---
## Step 6. 멀티턴 — `chat_history` (5.1 + Agent)

In [ ]:
print(executor.invoke({'input': '학칙에서 석사 수업연한 알려줘', 'chat_history': []})['output'])
print(executor.invoke({'input': '방금 말한 내용 한 줄로 다시', 'chat_history': []})['output'])

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

class ChatSession:
    def __init__(self, executor):
        self.executor = executor
        self.history = []

    def ask(self, question: str) -> str:
        result = self.executor.invoke({'input': question, 'chat_history': self.history})
        answer = result['output']
        self.history.extend([HumanMessage(content=question), AIMessage(content=answer)])
        return answer

session = ChatSession(executor)
print('1:', session.ask('학칙에서 석사 수업연한 알려줘'))
print('2:', session.ask('방금 답한 내용 한 줄로 다시'))

---
## Step 7. `input()` 대화 루프

In [ ]:
print('종료: exit')
while True:
    user_input = input('사용자: ').strip()
    if user_input.lower() in ('exit', 'quit', '종료'):
        break
    if not user_input:
        continue
    print('AI:', session.ask(user_input))

---
## Step 8. 나만의 Tool 만들기

`@tool`로 함수를 등록하고 `agent_tools`에 넣으면 Agent가 사용할 수 있습니다.

1. 함수 정의 (`@tool` + docstring)
2. `.invoke()`로 단독 테스트
3. `agent_tools`에 추가 → prompt 보강 → `executor` 재생성
4. Agent 질문으로 Tool 호출 확인

### 8.1 예시 — `celsius_to_fahrenheit`

섭씨 → 화씨 변환 Tool을 만들고 Agent에 연결합니다.

In [ ]:
from langchain_core.tools import tool

@tool
def celsius_to_fahrenheit(celsius: float) -> str:
    """섭씨 온도를 화씨로 변환합니다."""
    fahrenheit = celsius * 9 / 5 + 32
    return f'{celsius}°C = {fahrenheit:.1f}°F'

# 1) Tool 단독 실행
print(celsius_to_fahrenheit.name)
print(celsius_to_fahrenheit.invoke({'celsius': 25}))

In [ ]:
# 2) agent_tools에 추가 후 Agent 재생성
agent_tools = agent_tools + [celsius_to_fahrenheit]

prompt = ChatPromptTemplate.from_messages([
    ('system', '''너는 사용자를 돕는 AI입니다. 질문에 맞는 도구를 골라 사용하세요.

- samples/rag_samples PDF (학칙, news 등) → search_pdf_documents
- 최신 뉴스·시사·웹 정보 → duckduckgo_results_json
- 주가 → get_yf_stock_history
- 현재 시각 → get_current_time
- 섭씨→화씨 온도 변환 → celsius_to_fahrenheit

도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

print('등록된 Tool:', [t.name for t in agent_tools])

In [ ]:
# 3) Agent가 새 Tool을 호출하는지 확인
r = executor.invoke({'input': '섭씨 36.5도는 화씨로 몇 도야?', 'chat_history': []})
print(r['output'])

### 8.2 실습 — 나만의 Tool

원하는 기능의 Tool을 직접 만들고, Agent가 호출하는지 확인합니다.

- 예: 단위 변환, 문자열 처리, 간단한 계산, 로컬 파일 읽기 등
- docstring에 **언제 쓸지**를 적어야 Agent가 선택합니다.

In [ ]:
# TODO 1) @tool 함수 정의
# @tool
# def my_custom_tool(파라미터: 타입) -> str:
#     """Agent가 이 Tool을 언제 쓸지 설명하세요."""
#     ...

# TODO 2) 단독 실행 테스트
# print(my_custom_tool.invoke({...}))

# TODO 3) agent_tools에 추가 + prompt에 한 줄 안내 + executor 재생성
# agent_tools = agent_tools + [my_custom_tool]
# prompt = ChatPromptTemplate.from_messages([...])
# agent = create_tool_calling_agent(llm, agent_tools, prompt)
# executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

# TODO 4) Agent 질문으로 Tool 호출 확인
# r = executor.invoke({'input': '...', 'chat_history': []})
# print(r['output'])